# fak quickstart - fresh value demos

Build the current `fak` once, then run the case that matches what you are trying to prove.
The notebook defaults to `FAK_BRANCH=main`, refreshes an existing clone unless
`FAK_REFRESH=0`, and prints the exact commit and binary version it is about to run.

| Case | Use this when... | What it proves | Needs |
|---|---|---|---|
| **A. Policy floor** | you need a reviewable allow/deny boundary | `preflight --json` explains the winning rung; `attest` proves the manifest | CPU |
| **B. App/API drop-in** | you have a Python app, MCP adapter, or tool runner | `/v1/fak/adjudicate` gives a verdict before your code executes a tool | CPU |
| **C. Value measurement** | you want a number before wiring a model | offline benchmark catalog, vDSO transport A/B, and turn-tax report | CPU |
| **D. Real model gateway** | you want `fak` in front of a chat model | OpenAI-compatible client -> `fak serve` -> Ollama | GPU recommended |

No API key is needed for cases A-C. Case D uses a free Colab/Kaggle GPU and a small Ollama
model by default.

> _Generated by `tools/gen_notebooks.py` — edit the generator, not this `.ipynb`. CI guards drift + stale references via `--check`._

In [ ]:
# --- setup: work dir, repo coords, GPU ---
import os, shutil, subprocess, sys, time, json, urllib.request

WORK = os.environ.get("FAK_WORK") or ("/content" if os.path.isdir("/content")
        else "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/fak-demo")
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)

REPO   = os.environ.get("FAK_REPO", "anthony-chaudhary/fak")
BRANCH = os.environ.get("FAK_BRANCH", "main")  # pin a release tag here, e.g. v0.30.0

HAS_GPU = shutil.which("nvidia-smi") is not None and \
          subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("work dir :", WORK)
print(f"repo     : {REPO}@{BRANCH}")
print("GPU      :", "yes — Tier 1 ready" if HAS_GPU else "no — Tier 0 only (Runtime -> T4 for Tier 1)")

In [ ]:
# --- get a fresh `fak` binary plus the examples/testdata the demos use.
#     The Go module is the repository ROOT (go.mod), so the clone dir *is* the build dir.
#     By default an existing clone is refreshed to FAK_BRANCH. Set FAK_REFRESH=0 to reuse it.
import urllib.request, tarfile, platform, re

def sh(cmd, timeout=900):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, timeout=timeout)

def go_ok():
    try:
        out = subprocess.run(["go","version"], capture_output=True, text=True, check=True).stdout
        m = re.search(r"go(\d+)\.(\d+)", out)
        return bool(m) and (int(m.group(1)), int(m.group(2))) >= (1, 26)
    except Exception:
        return False

def latest_go_archive():
    arch = platform.machine().lower()
    goarch = {"x86_64":"amd64", "amd64":"amd64", "aarch64":"arm64", "arm64":"arm64"}.get(arch)
    if sys.platform.startswith("linux") and goarch:
        rels = json.load(urllib.request.urlopen("https://go.dev/dl/?mode=json", timeout=30))
        for rel in rels:
            m = re.match(r"go(\d+)\.(\d+)", rel.get("version", ""))
            if not m or (int(m.group(1)), int(m.group(2))) < (1, 26):
                continue
            for f in rel.get("files", []):
                if f.get("kind") == "archive" and f.get("os") == "linux" and f.get("arch") == goarch:
                    return rel["version"], f["filename"]
    return "go1.26.3", "go1.26.3.linux-amd64.tar.gz"

if not go_ok():
    GOV, GO = latest_go_archive()
    print("installing", GOV, "...")
    urllib.request.urlretrieve("https://go.dev/dl/" + GO, "/tmp/" + GO)
    shutil.rmtree("/usr/local/go", ignore_errors=True)
    with tarfile.open("/tmp/" + GO) as t: t.extractall("/usr/local")
os.environ["PATH"] = "/usr/local/go/bin:" + os.environ["PATH"]
print(subprocess.run(["go","version"], capture_output=True, text=True).stdout.strip())

# clone the public repo (anonymous; no token needed), then refresh to the requested ref
FAK_DIR = os.path.join(WORK, "fak")
if not os.path.isdir(os.path.join(FAK_DIR, ".git")):
    print(f"$ git clone --depth 1 -b {BRANCH} https://github.com/{REPO}.git  ->  {FAK_DIR}")
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,
                    f"https://github.com/{REPO}.git", FAK_DIR], check=True)
else:
    print("clone exists:", FAK_DIR)
    if os.environ.get("FAK_REFRESH", "1") != "0":
        sh(f'git -C "{FAK_DIR}" fetch --depth 1 origin "{BRANCH}"')
        sh(f'git -C "{FAK_DIR}" checkout -q --detach FETCH_HEAD')
    else:
        print("FAK_REFRESH=0, reusing the existing checkout without fetch")

stamp = subprocess.run(["git","-C",FAK_DIR,"log","-1","--format=%h %cI %s"],
                       capture_output=True, text=True, check=True).stdout.strip()
print("source  :", stamp)

# the module is the repo root, so build right here
FAK_EXE = "fak.exe" if os.name == "nt" else "fak"
FAK = os.path.join(FAK_DIR, FAK_EXE)
sh(f'cd "{FAK_DIR}" && go build -o "{FAK_EXE}" ./cmd/fak')
print("\nfak ->", FAK)
print(subprocess.run([FAK,"version"], capture_output=True, text=True).stdout.strip() or "built ok")

In [ ]:
# --- small helpers used by the cases below ---
POL = "examples/customer-support-readonly-policy.json"

def cmd(argv, cwd=FAK_DIR, check=True, timeout=180):
    p = subprocess.run(argv, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if check and p.returncode != 0:
        raise RuntimeError("command failed: " + " ".join(argv) + "\n" + p.stdout + p.stderr)
    return p

def fak(*args, check=True, timeout=180):
    return cmd([FAK, *args], check=check, timeout=timeout)

def fak_json(*args, timeout=180):
    p = fak(*args, timeout=timeout)
    return json.loads(p.stdout)

def clip(s, n=2400):
    s = s.strip()
    return s if len(s) <= n else s[:n] + "\n... clipped ..."

def post_json(base, path, payload, timeout=30):
    req = urllib.request.Request(base + path, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

def stop_process(name):
    p = globals().get(name)
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()

print("source :", cmd(["git","log","-1","--format=%h %cI %s"]).stdout.strip())
print("binary :")
print(fak("version").stdout.strip())

## Expected state for a CPU run

Run-all on a CPU-only runtime should finish Cases A-C and print these success signals:

- Case A: `refund_payment` is `DENY/POLICY_BLOCK`, `search_kb` is `ALLOW`, and the policy attestation passes with zero failed probes.
- Case B: local `fak serve` reports healthy, and `/v1/fak/adjudicate` denies `refund_payment` before your app executes it.
- Case C: `benchmarks list --offline` prints the zero-asset benchmark catalog, `bench` reports `PRIMARY GATE : pass`, and `turntax` reports 9 turns saved on the fixture.
- Case D: skipped on CPU-only runtimes; on a GPU runtime it should serve `qwen2.5:1.5b` through the OpenAI SDK.

The cells below include assertions for the stable CPU expectations, so a stale policy,
missing verb, or broken HTTP adjudication path fails where it happens.

## Case A - policy floor you can ship

This is the no-model proof. `preflight` runs the same adjudication fold a served call uses,
and `attest` turns the policy file into a re-checkable proof obligation.

In [ ]:
# A1. Three structural calls: explicit deny, prefix allow, fail-closed unknown.
cases = [
    ("explicit deny", "refund_payment", {}, "DENY", "POLICY_BLOCK"),
    ("prefix allow", "search_kb", {"query": "refund policy"}, "ALLOW", ""),
    ("fail-closed unknown", "__not_in_policy__", {}, "DENY", "DEFAULT_DENY"),
]
for label, tool, args, want_kind, want_reason in cases:
    d0 = fak_json("preflight", "--policy", POL, "--tool", tool, "--args", json.dumps(args),
                  "--json")
    got = (d0["verdict"], d0.get("reason", ""))
    assert d0["verdict"] == want_kind, (label, got, want_kind)
    if want_reason:
        assert d0.get("reason") == want_reason, (label, got, want_reason)
    print(f"{label:<20} {tool:<22} -> {d0['verdict']} {d0.get('reason','')}")

# A2. Explain the exact winning rung without logging raw args.
d = fak_json("preflight", "--policy", POL, "--tool", "refund_payment", "--args", "{}",
             "--json")
print("\nwinning verdict:", d["verdict"], d.get("reason", ""), "by", d.get("by", ""))
print("explanation    :", d["explanation"])
print("\nrung trace:")
for r in d["rungs"]:
    mark = "WIN" if r.get("winner") else "   "
    print(f"{mark} [{r['index']}] {r['rung']:<26} {r.get('kind','ELIDED'):<9} {r.get('reason','')}")

In [ ]:
# A3. Prove the whole manifest, not just one cherry-picked call.
att = fak_json("attest", "--policy", POL, "--quiet")
assert att["summary"]["pass"] is True, att["summary"]
assert att["summary"]["failed"] == 0, att["summary"]
print("attestation schema:", att["schema"])
print("policy sha256     :", att["policy"]["sha256"][:16] + "...")
print("probes            :", att["summary"]["probes"])
print("passed / failed   :", att["summary"]["passed"], "/", att["summary"]["failed"])
print("floor proven      :", att["summary"]["pass"])

## Case B - app/API drop-in

Use this path when your own application executes tools. The application asks `fak` for a
verdict first, then executes only the calls that survive. No chat model or GPU is involved.

In [ ]:
# Start a local fak HTTP gateway for adjudication-only calls.
stop_process("fak_api_proc")
API_ADDR = os.environ.get("FAK_API_ADDR", "127.0.0.1:8765")
API = "http://" + API_ADDR
fak_api_proc = subprocess.Popen(
    [FAK, "serve", "--addr", API_ADDR, "--policy", POL, "--model", "quickstart"],
    cwd=FAK_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        with urllib.request.urlopen(API + "/healthz", timeout=2) as r:
            print("gateway:", r.read().decode().strip())
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("fak serve did not become healthy on " + API)

In [ ]:
# The same endpoint accepts JSON objects and OpenAI-style JSON-string arguments.
api_cases = [
    ("read-only app action", {"tool": "search_kb", "arguments": {"query": "refund policy"},
                              "read_only": True}),
    ("blocked business action", {"tool": "refund_payment", "arguments": {"amount": 100}}),
    ("OpenAI function args", {"tool": "refund_payment",
                              "arguments": json.dumps({"amount": 100})}),
]
for label, payload in api_cases:
    r = post_json(API, "/v1/fak/adjudicate", payload)
    v = r["verdict"]
    if payload["tool"] == "refund_payment":
        assert v["kind"] == "DENY" and v.get("reason") == "POLICY_BLOCK", v
    if payload["tool"] == "search_kb":
        assert v["kind"] == "ALLOW", v
    print(f"{label:<24} -> {v['kind']:<5} {v.get('reason','')} {v.get('disposition','')}")

print("\nThis was adjudicate-only: your application still owns execution of allowed tools.")

## Case C - value measurement without a model

Before you spend GPU time, ask what can be measured offline. The catalog names the benchmark
surfaces, then two small runs show the transport/value axes on committed fixtures.

In [ ]:
# Discover the zero-asset benchmark surfaces.
print(clip(fak("benchmarks", "list", "--offline").stdout, 3600))

In [ ]:
# Run the short transport benchmark and a turn-tax report on frozen fixtures.
print("== transport A/B ==")
bench_out = fak("bench", "--suite", "tau2-smoke", "--baseline-n", "5",
                "--out", "quickstart-bench.json", timeout=240).stdout
assert "PRIMARY GATE" in bench_out and "pass" in bench_out, bench_out
print(clip(bench_out, 2200))

print("\n== turn-tax A/B ==")
turntax_out = fak("turntax", "--suite", "turntax-airline",
                  "--out", "quickstart-turntax.json", timeout=240).stdout
assert "turns saved" in turntax_out and ": 9" in turntax_out, turntax_out
print(clip(turntax_out, 2600))

## Case D - real model gateway

If the runtime has a GPU, this starts Ollama with a small model and puts `fak serve` in front
of it. The client is the standard OpenAI Python SDK pointed at `http://127.0.0.1:8080/v1`.

In [ ]:
# D1. Start Ollama and pull a small model that fits a free T4.
MODEL = os.environ.get("FAK_MODEL", "qwen2.5:1.5b")

if not HAS_GPU:
    print("no GPU - skipping Case D. Switch Runtime -> Change runtime type -> T4 and run again.")
else:
    if shutil.which("ollama") is None:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    if not globals().get("ollama_proc") or ollama_proc.poll() is not None:
        ollama_proc = subprocess.Popen(["ollama", "serve"],
                                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    print("ollama up; pulling", MODEL, "(one-time, cached) ...")
    subprocess.run(["ollama", "pull", MODEL], check=True)
    print("model ready:", MODEL)

In [ ]:
# D2. Start fak in front of Ollama.
if HAS_GPU:
    stop_process("fak_model_proc")
    fak_model_proc = subprocess.Popen(
        [FAK, "serve", "--addr", "127.0.0.1:8080",
         "--base-url", "http://127.0.0.1:11434/v1",
         "--model", MODEL, "--policy", POL],
        cwd=FAK_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            with urllib.request.urlopen("http://127.0.0.1:8080/healthz", timeout=2) as r:
                print("fak serve:", r.read().decode().strip())
                break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("fak serve did not become healthy on :8080")

In [ ]:
# D3. Drive it with the OpenAI SDK.
if HAS_GPU:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
    from openai import OpenAI
    client = OpenAI(base_url="http://127.0.0.1:8080/v1", api_key="not-needed")
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[{"role": "user", "content": "In one sentence, what is a syscall?"}],
    )
    print("model through fak:", resp.choices[0].message.content)
    models = json.load(urllib.request.urlopen("http://127.0.0.1:8080/v1/models", timeout=10))
    print("\n/v1/models:", json.dumps(models, indent=2)[:900])

## Where to go next

- **Tier 2 - the fused in-kernel model** (`fak serve --engine inkernel`, real SmolLM2 via
  `scripts/fetch-model.sh`): see **`fak-inkernel.ipynb`** in this folder, or
  `GETTING-STARTED.md` section 4.
- **Policy authoring:** `POLICY.md` · **Architecture:** `ARCHITECTURE.md`
- **What's real vs. simulated vs. stub:** `CLAIMS.md` (every claim machine-tagged)

*Cleanup* - background servers die when the notebook runtime recycles. To stop them now:
`for p in [fak_api_proc, fak_model_proc, ollama_proc]: p.terminate()`